In [1]:
import sys
import pandas as pd

print(sys.executable)

/Users/dhk/Python-Group-Project/.venv/bin/python


In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path("data/raw")

print("Notebook folder:", Path.cwd())
print("Data files:", [p.name for p in data_dir.glob("*.parquet")])

Notebook folder: /Users/dhk/Python-Group-Project
Data files: []


In [3]:
print([p.name for p in data_dir.glob("*.parquet")])

['filings.parquet', 'prices.parquet', 'news.parquet']


In [4]:
prices_raw = pd.read_parquet(data_dir / "prices.parquet")
news_raw = pd.read_parquet(data_dir / "news.parquet")
filings_raw = pd.read_parquet(data_dir / "filings.parquet")

print("Prices:", prices_raw.shape)
print("News:", news_raw.shape)
print("Filings:", filings_raw.shape)

Prices: (2656, 9)
News: (3888, 4)
Filings: (73, 6)


In [5]:
prices_raw.head()

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


In [6]:
prices_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2656 entries, 0 to 2655
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         2656 non-null   int32  
 1   symbol     2656 non-null   str    
 2   date       2656 non-null   object 
 3   open       2656 non-null   float64
 4   high       2656 non-null   float64
 5   low        2656 non-null   float64
 6   close      2656 non-null   float64
 7   adj_close  2656 non-null   float64
 8   volume     2656 non-null   int64  
dtypes: float64(5), int32(1), int64(1), object(1), str(1)
memory usage: 187.2+ KB


In [7]:
prices_raw["symbol"].value_counts(dropna=False)

symbol
AAPL     332
ADBE     332
AMZN     332
GOOGL    332
META     332
MSFT     332
NVDA     332
TSLA     332
Name: count, dtype: int64

In [8]:
print("Exact duplicate rows:", prices_raw.duplicated().sum())

print("Repeated IDs:", prices_raw["id"].duplicated().sum())

print(
    "Repeated company-date pairs:",
    prices_raw.duplicated(subset=["symbol", "date"]).sum()
)

Exact duplicate rows: 0
Repeated IDs: 0
Repeated company-date pairs: 0


In [9]:
dates_check = pd.to_datetime(prices_raw["date"], errors="coerce")

print("Original date value type:", type(prices_raw["date"].iloc[0]))
print("Failed date conversions:",
      (prices_raw["date"].notna() & dates_check.isna()).sum())
print("Earliest date:", dates_check.min())
print("Latest date:", dates_check.max())

Original date value type: <class 'datetime.date'>
Failed date conversions: 0
Earliest date: 2024-12-02 00:00:00
Latest date: 2026-03-31 00:00:00


In [10]:
price_columns = ["open", "high", "low", "close", "adj_close"]

print("Zero or negative prices by column:")
print(prices_raw[price_columns].le(0).sum())

print("Negative volume:", prices_raw["volume"].lt(0).sum())
print("Zero volume:", prices_raw["volume"].eq(0).sum())

Zero or negative prices by column:
open         0
high         0
low          0
close        0
adj_close    0
dtype: int64
Negative volume: 0
Zero volume: 0


| Rule | Reason | Verification |
|---|---|---|
| Keep only AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA | Match the revised project scope | Exclude 332 ADBE rows; expect 2,324 remaining rows |
| Convert `date` to pandas datetime | Support date filtering and time-series calculations | No failed conversions; preserve calendar dates |
| Preserve original IDs and numeric values | No issue found requiring changes so far | Compare retained values with the original |
| Do not remove duplicates or fill missing values | Checks found no duplicates or nulls in prices | Repeat checks after transformation |
| Sort by symbol and date | Put each company’s observations in chronological order | Inspect the sorted output |

In [11]:
selected_symbols = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"
]

prices_clean = prices_raw.loc[
    prices_raw["symbol"].isin(selected_symbols)
].copy()

prices_clean["date"] = pd.to_datetime(
    prices_clean["date"], errors="raise"
)

prices_clean = prices_clean.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

In [12]:
print("Original rows:", len(prices_raw))
print("Excluded rows:", (~prices_raw["symbol"].isin(selected_symbols)).sum())
print("Final rows:", len(prices_clean))

print("\nRows per company:")
print(prices_clean["symbol"].value_counts())

print("\nMissing values:")
print(prices_clean.isna().sum())

print("\nRepeated company-date pairs:",
      prices_clean.duplicated(["symbol", "date"]).sum())

Original rows: 2656
Excluded rows: 332
Final rows: 2324

Rows per company:
symbol
AAPL     332
AMZN     332
GOOGL    332
META     332
MSFT     332
NVDA     332
TSLA     332
Name: count, dtype: int64

Missing values:
id           0
symbol       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

Repeated company-date pairs: 0


In [13]:
original_retained = prices_raw.loc[
    prices_raw["symbol"].isin(selected_symbols)
].sort_values(["symbol", "date"]).reset_index(drop=True)

unchanged_columns = [
    "id", "symbol", "open", "high", "low",
    "close", "adj_close", "volume"
]

assert prices_clean[unchanged_columns].equals(
    original_retained[unchanged_columns]
)

assert prices_clean["date"].dt.date.equals(
    original_retained["date"]
)

print("Retained values and calendar dates are unchanged.")

Retained values and calendar dates are unchanged.


In [14]:
news_raw.info()
display(news_raw.head(3))

filings_raw.info()
display(filings_raw.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 3888 entries, 0 to 3887
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          3888 non-null   int32 
 1   symbol      3888 non-null   str   
 2   date        3888 non-null   object
 3   highlights  3888 non-null   str   
dtypes: int32(1), object(1), str(2)
memory usage: 17.8+ MB


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a..."
2,3,AAPL,2024-12-03,"demonstrated strong financial performance, wit..."


<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             73 non-null     int32 
 1   symbol         73 non-null     str   
 2   date           73 non-null     object
 3   document_type  73 non-null     str   
 4   mda_content    73 non-null     str   
 5   risk_content   73 non-null     str   
dtypes: int32(1), object(1), str(4)
memory usage: 7.7+ MB


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


In [15]:
for name, df in [("News", news_raw), ("Filings", filings_raw)]:
    dates = pd.to_datetime(df["date"], errors="coerce")

    print(f"\n{name}")
    print("Rows per company:")
    print(df["symbol"].value_counts(dropna=False))
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Repeated IDs:", df["id"].duplicated().sum())
    print("Repeated company-date pairs:",
          df.duplicated(["symbol", "date"]).sum())
    print("Failed date conversions:",
          (df["date"].notna() & dates.isna()).sum())
    print("Date range:", dates.min(), "to", dates.max())


News
Rows per company:
symbol
AAPL     486
ADBE     486
AMZN     486
GOOGL    486
META     486
MSFT     486
NVDA     486
TSLA     486
Name: count, dtype: int64
Exact duplicate rows: 0
Repeated IDs: 0
Repeated company-date pairs: 0
Failed date conversions: 0
Date range: 2024-12-01 00:00:00 to 2026-03-31 00:00:00

Filings
Rows per company:
symbol
ADBE     10
AAPL      9
AMZN      9
GOOGL     9
META      9
MSFT      9
NVDA      9
TSLA      9
Name: count, dtype: int64
Exact duplicate rows: 0
Repeated IDs: 0
Repeated company-date pairs: 0
Failed date conversions: 0
Date range: 2024-01-16 00:00:00 to 2026-03-25 00:00:00


In [16]:
print("Blank news highlights:",
      news_raw["highlights"].str.strip().eq("").sum())

for column in ["mda_content", "risk_content"]:
    print(f"Blank {column}:",
          filings_raw[column].str.strip().eq("").sum())

print("\nFiling types:")
print(filings_raw["document_type"].value_counts(dropna=False))

Blank news highlights: 0
Blank mda_content: 0
Blank risk_content: 0

Filing types:
document_type
10-Q    51
10-K    22
Name: count, dtype: int64


In [17]:
import html

original_text = filings_raw.loc[0, "mda_content"]

print("BEFORE:")
print(original_text[:250])

print("\nAFTER:")
print(html.unescape(original_text)[:250])

BEFORE:
 Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operations 

This Item and other sections of this Quarterly Report on Form 10-Q (&#8220;Form 10-Q&#8221;) contain forward-looking statements, within the meaning

AFTER:
 Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations 

This Item and other sections of this Quarterly Report on Form 10-Q (“Form 10-Q”) contain forward-looking statements, within the meaning of the Private Se


| Rule | Reason | Check |
|---|---|---|
| Keep the seven selected companies | Match the project scope | Exclude 486 ADBE news rows and 10 ADBE filing rows |
| Convert dates to pandas datetime | Support date filtering | Preserve calendar dates; no conversion failures |
| Decode HTML entities in filing text once | Make encoded punctuation readable | Count changed records and inspect before/after examples |
| Preserve news text, IDs, and filing types | No correction is currently justified | Compare against original retained records |
| Keep all available dates | Earlier filings and nontrading-day news provide context | No date-based row removal |

In [18]:
import html

news_clean = news_raw.loc[
    news_raw["symbol"].isin(selected_symbols)
].copy()

filings_clean = filings_raw.loc[
    filings_raw["symbol"].isin(selected_symbols)
].copy()

news_clean["date"] = pd.to_datetime(
    news_clean["date"], errors="raise"
)

filings_clean["date"] = pd.to_datetime(
    filings_clean["date"], errors="raise"
)

for column in ["mda_content", "risk_content"]:
    before = filings_clean[column].copy()
    filings_clean[column] = before.map(html.unescape)

    print(f"{column}: {(before != filings_clean[column]).sum()} records changed")

news_clean = news_clean.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

filings_clean = filings_clean.sort_values(
    ["symbol", "date", "id"]
).reset_index(drop=True)

mda_content: 63 records changed
risk_content: 58 records changed


In [19]:
for name, raw, clean in [
    ("News", news_raw, news_clean),
    ("Filings", filings_raw, filings_clean)
]:
    print(f"\n{name}")
    print("Original rows:", len(raw))
    print("Excluded rows:", (~raw["symbol"].isin(selected_symbols)).sum())
    print("Final rows:", len(clean))
    print("Exact duplicates:", clean.duplicated().sum())
    print("Repeated IDs:", clean["id"].duplicated().sum())
    print("Missing values:")
    print(clean.isna().sum())


News
Original rows: 3888
Excluded rows: 486
Final rows: 3402
Exact duplicates: 0
Repeated IDs: 0
Missing values:
id            0
symbol        0
date          0
highlights    0
dtype: int64

Filings
Original rows: 73
Excluded rows: 10
Final rows: 63
Exact duplicates: 0
Repeated IDs: 0
Missing values:
id               0
symbol           0
date             0
document_type    0
mda_content      0
risk_content     0
dtype: int64


In [20]:
for raw, clean, columns in [
    (news_raw, news_clean, ["symbol", "highlights"]),
    (filings_raw, filings_clean, ["symbol", "document_type"])
]:
    original = raw.set_index("id").loc[clean["id"]]
    cleaned = clean.set_index("id")

    assert original[columns].equals(cleaned[columns])
    assert pd.to_datetime(original["date"]).equals(cleaned["date"])

print("Retained IDs, unchanged fields, and calendar dates verified.")

Retained IDs, unchanged fields, and calendar dates verified.


In [21]:
invalid_ohlc = (
    (prices_raw["high"] < prices_raw["low"])
    | (prices_raw["open"] > prices_raw["high"])
    | (prices_raw["open"] < prices_raw["low"])
    | (prices_raw["close"] > prices_raw["high"])
    | (prices_raw["close"] < prices_raw["low"])
)

print("Inconsistent price rows:", invalid_ohlc.sum())
display(prices_raw.loc[invalid_ohlc])

Inconsistent price rows: 0


,id,symbol,date,open,high,low,close,adj_close,volume


| Calculation | Rule |
|---|---|
| Daily return | Current adjusted close ÷ previous observed adjusted close − 1, calculated separately for each company |
| Full-period return | Last adjusted close ÷ first adjusted close − 1 |
| Daily volatility | Sample standard deviation of daily returns |
| Missing returns | Keep the first return for each company missing; there is no preceding observation |
| Missing group keys | Require nonmissing company symbols and dates before calculating |

In [22]:
prices_analysis = prices_clean.sort_values(
    ["symbol", "date"]
).copy()

prices_analysis["daily_return"] = (
    prices_analysis.groupby("symbol")["adj_close"]
    .pct_change(fill_method=None)
)

print("Missing daily returns:",
      prices_analysis["daily_return"].isna().sum())

display(
    prices_analysis[
        ["symbol", "date", "adj_close", "daily_return"]
    ].head()
)

Missing daily returns: 7


,symbol,date,adj_close,daily_return
0,AAPL,2024-12-02,238.288300,NaN
1,AAPL,2024-12-03,241.331665,0.012772
2,AAPL,2024-12-04,241.689728,0.001484
3,AAPL,2024-12-05,241.719559,0.000123
4,AAPL,2024-12-06,241.520630,-0.000823


In [23]:
metrics = prices_analysis.groupby("symbol").agg(
    n_prices=("adj_close", "count"),
    n_returns=("daily_return", "count"),
    first_price=("adj_close", "first"),
    last_price=("adj_close", "last"),
    daily_volatility=("daily_return", "std"),
    average_volume=("volume", "mean"),
    n_volume=("volume", "count")
)

metrics["period_return"] = (
    metrics["last_price"] / metrics["first_price"] - 1
)

display(metrics)

,n_prices,n_returns,first_price,last_price,daily_volatility,average_volume,n_volume,period_return
symbol,,,,,,,,
AAPL,332,331,238.288300,253.789993,0.019191,5.256276e+07,332,0.065054
AMZN,332,331,210.710007,208.270004,0.021142,4.513671e+07,332,-0.011580
GOOGL,332,331,170.505417,287.559998,0.019884,3.557789e+07,332,0.686515
META,332,331,589.985046,572.130005,0.024066,1.538954e+07,332,-0.030264
MSFT,332,331,426.849762,370.170013,0.016331,2.475262e+07,332,-0.132786
NVDA,332,331,138.575043,174.399994,0.029398,2.118627e+08,332,0.258524
TSLA,332,331,357.089996,371.750000,0.037553,9.062856e+07,332,0.041054


In [24]:
original_volume = int(prices_raw["volume"].sum())

excluded_volume = int(
    prices_raw.loc[
        ~prices_raw["symbol"].isin(selected_symbols), "volume"
    ].sum()
)

cleaned_volume = int(prices_clean["volume"].sum())

print("Original volume:", original_volume)
print("Excluded ADBE volume:", excluded_volume)
print("Cleaned volume:", cleaned_volume)

assert original_volume - excluded_volume == cleaned_volume
print("Volume reconciliation passed.")

Original volume: 159456800900
Excluded ADBE volume: 1454421600
Cleaned volume: 158002379300
Volume reconciliation passed.


In [25]:
check_price = prices_raw.loc[
    prices_raw["symbol"].eq("AAPL")
].sort_values("date").iloc[0]

check_excluded = prices_raw.loc[
    prices_raw["symbol"].eq("ADBE")
].sort_values("date").iloc[0]

check_news = news_raw.loc[
    news_raw["symbol"].eq("AAPL")
].sort_values("date").iloc[0]

check_filing = filings_raw.loc[
    filings_raw["symbol"].eq("AAPL")
    & filings_raw["mda_content"].str.contains("&#8217;", regex=False)
].sort_values("date").iloc[0]

check_return = prices_raw.loc[
    prices_raw["symbol"].eq("AMZN")
].sort_values("date").iloc[0]

display(check_price)
display(check_excluded)
display(check_news)
print("Filing ID:", check_filing["id"])
print(check_filing["mda_content"][:250])
display(check_return)

id                    1
symbol             AAPL
date         2024-12-02
open         237.270004
high         240.789993
low          237.160004
close        239.589996
adj_close      238.2883
volume         48137100
Name: 0, dtype: object

id                  333
symbol             ADBE
date         2024-12-02
open         512.849976
high          518.77002
low          511.529999
close        516.200012
adj_close    516.200012
volume          2006900
Name: 332, dtype: object

id                                                            1
symbol                                                     AAPL
date                                                 2024-12-01
highlights    Apple’s December 1, 2024 coverage centers on i...
Name: 0, dtype: object

Filing ID: 1
 Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operations 

This Item and other sections of this Quarterly Report on Form 10-Q (&#8220;Form 10-Q&#8221;) contain forward-looking statements, within the meaning


id                  665
symbol             AMZN
date         2024-12-02
open         209.960007
high         212.990005
low          209.509995
close        210.710007
adj_close    210.710007
volume         39523200
Name: 664, dtype: object

| Record | Expected result | Reason |
|---|---|---|
| First AAPL price record | Retained; same ID, price, volume, and calendar date | AAPL is in scope; date conversion should preserve the date |
| First ADBE price record | Excluded | ADBE is outside the seven-company scope |
| First AAPL news record | Retained with unchanged highlights and date | News can be useful even on a day without a price observation |
| Selected AAPL filing | Retained; `&#8217;` becomes `’` | Decode encoded punctuation for readability |
| First AMZN price record | Retained; its calculated return is missing | Returns must restart for each company, without using another company’s price |

| Record | ID |
|---|---:|
| AAPL price | 1 |
| ADBE price | 333 |
| AAPL news | 1 |
| AAPL filing | 1 |
| First AMZN price | 665 |

In [26]:
# Get the retained records using their original IDs
actual_price = prices_clean.set_index("id").loc[check_price["id"]]
actual_news = news_clean.set_index("id").loc[check_news["id"]]
actual_filing = filings_clean.set_index("id").loc[check_filing["id"]]
actual_return = prices_analysis.set_index("id").loc[check_return["id"]]

# 1. AAPL price: preserve all numeric values and the calendar date
numeric_columns = ["open", "high", "low", "close", "adj_close", "volume"]

price_pass = (
    actual_price["symbol"] == check_price["symbol"]
    and actual_price["date"].date() == check_price["date"]
    and all(
        actual_price[c] == check_price[c]
        for c in numeric_columns
    )
)

# 2. ADBE price: excluded from the analysis scope
excluded_pass = not prices_clean["id"].eq(check_excluded["id"]).any()

# 3. AAPL news: preserve text and calendar date
news_pass = (
    actual_news["symbol"] == check_news["symbol"]
    and actual_news["date"].date() == check_news["date"]
    and actual_news["highlights"] == check_news["highlights"]
)

# 4. Filing: check the expected decoded phrase
filing_pass = (
    "Management&#8217;s" in check_filing["mda_content"]
    and "Management’s" in actual_filing["mda_content"]
    and "Management&#8217;s" not in actual_filing["mda_content"]
)

# 5. AMZN: the first observation has no prior return
return_pass = (
    actual_return["symbol"] == "AMZN"
    and pd.isna(actual_return["daily_return"])
)

record_checks = pd.DataFrame({
    "record": [
        "Prices ID 1",
        "Prices ID 333",
        "News ID 1",
        "Filings ID 1",
        "Prices ID 665"
    ],
    "expected": [
        "Retained; numeric values and date unchanged",
        "Excluded",
        "Retained; highlights and date unchanged",
        "Management&#8217;s becomes Management’s",
        "Retained; first AMZN return is missing"
    ],
    "actual": [
        f"Date={actual_price['date'].date()}, "
        f"adj_close={actual_price['adj_close']}, "
        f"volume={actual_price['volume']}",
        f"Rows found: {prices_clean['id'].eq(check_excluded['id']).sum()}",
        f"Date={actual_news['date'].date()}, "
        f"text unchanged={actual_news['highlights'] == check_news['highlights']}",
        actual_filing["mda_content"][:100],
        f"daily_return={actual_return['daily_return']}"
    ],
    "matches_expected": [
        price_pass, excluded_pass, news_pass, filing_pass, return_pass
    ]
})

display(record_checks)
assert record_checks["matches_expected"].all()

,record,expected,actual,matches_expected
0,Prices ID 1,Retained; numeric values and date unchanged,"Date=2024-12-02, adj_close=238.28829956054688,...",True
1,Prices ID 333,Excluded,Rows found: 0,True
2,News ID 1,Retained; highlights and date unchanged,"Date=2024-12-01, text unchanged=True",True
3,Filings ID 1,Management&#8217;s becomes Management’s,Item 2. Management’s Discussion and Analysis ...,True
4,Prices ID 665,Retained; first AMZN return is missing,daily_return=nan,True


In [27]:
import math

aapl_original = prices_raw.loc[
    prices_raw["symbol"].eq("AAPL")
].sort_values("date")

previous_price = aapl_original.iloc[0]["adj_close"]
current_price = aapl_original.iloc[1]["adj_close"]

expected_return = current_price / previous_price - 1

actual_daily_return = prices_analysis.set_index("id").loc[
    aapl_original.iloc[1]["id"], "daily_return"
]

print("Previous adjusted close:", previous_price)
print("Current adjusted close:", current_price)
print("Expected return:", expected_return)
print("Actual return:", actual_daily_return)

assert math.isclose(
    expected_return, actual_daily_return,
    rel_tol=1e-12, abs_tol=1e-12
)

print("Numeric return check passed.")

Previous adjusted close: 238.28829956054688
Current adjusted close: 241.3316650390625
Expected return: 0.012771778908692744
Actual return: 0.012771778908692744
Numeric return check passed.


In [28]:
record_checks["original"] = [
    f"Date={check_price['date']}, "
    f"adj_close={check_price['adj_close']}, "
    f"volume={check_price['volume']}",
    f"Symbol={check_excluded['symbol']}, date={check_excluded['date']}",
    f"Date={check_news['date']}; {check_news['highlights'][:150]}",
    check_filing["mda_content"][:100],
    f"First AMZN observation: date={check_return['date']}, "
    f"adj_close={check_return['adj_close']}"
]

record_checks["reason"] = [
    "AAPL is in scope; preserve numeric values and calendar date.",
    "ADBE is outside the seven-company scope.",
    "Keep news even when there is no same-day price observation.",
    "Decode HTML punctuation for readability.",
    "The first observation has no previous price within its company."
]

record_checks = record_checks[
    ["record", "original", "expected", "reason", "actual", "matches_expected"]
]

In [29]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

prices_clean.to_parquet(output_dir / "prices_clean.parquet", index=False)
news_clean.to_parquet(output_dir / "news_clean.parquet", index=False)
filings_clean.to_parquet(output_dir / "filings_clean.parquet", index=False)

prices_analysis.to_parquet(
    output_dir / "prices_analysis.parquet", index=False
)

metrics.to_csv(output_dir / "stock_metrics.csv")
record_checks.to_csv(output_dir / "record_checks.csv", index=False)

print("Cleaned tables and evidence saved.")

Cleaned tables and evidence saved.


In [30]:
sample_dir = Path("samples")
sample_dir.mkdir(exist_ok=True)

price_sample = pd.concat([
    prices_analysis.groupby("symbol").head(2),
    prices_analysis.groupby("symbol").tail(2)
]).drop_duplicates("id").sort_values(["symbol", "date"])

news_sample = news_clean.groupby("symbol").head(1)
filings_sample = filings_clean.groupby("symbol").head(1)

price_sample.to_csv(sample_dir / "prices_sample.csv", index=False)
news_sample.to_csv(sample_dir / "news_sample.csv", index=False)
filings_sample.to_csv(sample_dir / "filings_sample.csv", index=False)

print("Price sample:", len(price_sample))
print("News sample:", len(news_sample))
print("Filings sample:", len(filings_sample))
print("Total:", len(price_sample) + len(news_sample) + len(filings_sample))

Price sample: 28
News sample: 7
Filings sample: 7
Total: 42
